In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt

In [ ]:
local = "sm"
year = "2026"
month = "01"

In [ ]:
df = pd.read_excel(f"data/{local}-sales-{year}-{month}.xls", sheet_name="Adiciones")
df_expenses = pd.read_excel(f"data/{local}-gastos-{year}-{month}.xlsx", sheet_name="Gastos", skiprows=3)

In [ ]:
df = df[df["Cancelada"] == "No"]

skipped_columns = [
    "Id. Venta",
    "Creación",
    "Producto",
    "Categoría",
    "Cantidad",
    "Precio",
    "Costo base",
    "Costo modificadores",
    "Costo total",
    "Creada por",
]
df = df[skipped_columns]

# if Producto column is 'Duo Familiar (2pizzas)' set Costo modificadores to 0
df.loc[df["Producto"] == "Duo Familiar (2pizzas)", "Costo modificadores"] = 0
df["Costo total"] = df["Costo base"] + df["Costo modificadores"]

df["created_at"] = pd.to_datetime(df["Creación"], errors='coerce')
df.drop(columns=["Creación"], inplace=True)
df.sort_values(by="created_at", inplace=True)

In [ ]:
df

In [ ]:
# Group by day and hour, count sales, and sum Precio
df['date'] = df['created_at'].dt.date
df['hour'] = df['created_at'].dt.hour

# substitute hour 0 with 24 to represent midnight
df['hour'] = df['hour'].replace(0, 24)

# Count sales per hour
sales_count = df.groupby(['date', 'hour']).size().reset_index(name='sales_count')

# Sum Precio per hour
precio_sum = df.groupby(['date', 'hour'])['Precio'].sum().reset_index(name='total_precio')

# Combine results
result = pd.merge(sales_count, precio_sum, on=['date', 'hour'])

print("Sales grouped by day and hour:")
print(result.head(20))

In [ ]:
result

In [ ]:
# Create charts for sales analysis

# 1. Chart showing amount of sales per hour per day
plt.figure(figsize=(15, 10))

# Group by hour to get total sales per hour
hourly_sales = result.groupby('hour')['sales_count'].sum()
hourly_revenue = result.groupby('hour')['total_precio'].sum()

plt.subplot(2, 1, 1)
plt.plot(hourly_sales.index, hourly_sales.values, marker='o', linewidth=2, markersize=6)
plt.title('Sales Count by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Total Sales Count')
plt.grid(True, alpha=0.3)
plt.xticks(range(10, 24))
plt.tight_layout()

# 2. Chart showing sum of price by hour per day
plt.subplot(2, 1, 2)
plt.plot(hourly_revenue.index, hourly_revenue.values, marker='o', linewidth=2, markersize=6, color='orange')
plt.title('Total Revenue by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Total Revenue ($)')
plt.grid(True, alpha=0.3)
plt.xticks(range(10, 24))
plt.tight_layout()
plt.show()

# Analysis: Find peak hours and best sales days
print("\n=== ANALYSIS RESULTS ===")

# Peak hours analysis
hourly_sales = result.groupby('hour')['sales_count'].sum()
peak_sales_hour = hourly_sales.idxmax()
peak_sales_count = hourly_sales.max()

hourly_revenue = result.groupby('hour')['total_precio'].sum()
peak_revenue_hour = hourly_revenue.idxmax()
peak_revenue_amount = hourly_revenue.max()

print(f"Peak sales hour: {peak_sales_hour}:00 with {peak_sales_count} sales")
print(f"Peak revenue hour: {peak_revenue_hour}:00 with ${peak_revenue_amount:,}")

# Best sales days
daily_sales = result.groupby('date')['sales_count'].sum()
best_sales_day = daily_sales.idxmax()
best_sales_count = daily_sales.max()

daily_revenue = result.groupby('date')['total_precio'].sum()
best_revenue_day = daily_revenue.idxmax()
best_revenue_amount = daily_revenue.max()

print(f"\nBest sales day: {best_sales_day} with {best_sales_count} sales")
print(f"Best revenue day: {best_revenue_day} with ${best_revenue_amount:,}")

# Hours with low sales (potential optimization opportunities)
low_sales_hours = hourly_sales[hourly_sales < hourly_sales.mean() * 0.5]
print(f"\nHours with low sales (<50% of average): {list(low_sales_hours.index)}")

# Summary
print(f"\n=== SUMMARY ===")
print(f"- Peak sales occur at {peak_sales_hour}:00")
print(f"- Peak revenue occurs at {peak_revenue_hour}:00")
print(f"- Best sales day was {best_sales_day}")
print(f"- Best revenue day was {best_revenue_day}")
print(f"- Consider focusing resources during peak hours ({peak_sales_hour}:00-{peak_revenue_hour}:00)")

In [ ]:
# Import plotly for interactive charts
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Interactive chart grouped by Id. Venta, year, month, and hour
# Add year, month, and day columns for better grouping
df['year'] = df['created_at'].dt.year
df['month'] = df['created_at'].dt.month
df['day'] = df['created_at'].dt.day

# Group by Id. Venta, year, month, and hour to see best times to sell
sales_by_time = df.groupby(['Id. Venta', 'year', 'month', 'hour']).agg({
    'Precio': 'sum',
    'Cantidad': 'sum'
}).reset_index()

# Create interactive heatmap for best selling times
fig1 = px.density_heatmap(
    sales_by_time, 
    x='hour', 
    y='month', 
    z='Precio',
    animation_frame='year',
    title='Best Selling Times by Hour and Month (Interactive)',
    color_continuous_scale='Viridis',
    labels={'hour': 'Hour of Day', 'month': 'Month', 'Precio': 'Total Revenue'}
)

fig1.update_layout(
    xaxis=dict(tickmode='linear', tick0=0, dtick=1),
    hovermode='x unified'
)

fig1.show()

# 2. Interactive chart showing sales by each day of the month
# Group by day of month to see daily patterns
daily_sales = df.groupby(['year', 'month', 'day']).agg({
    'Id. Venta': 'count',
    'Precio': 'sum'
}).reset_index()

# Create interactive line chart for daily sales
fig2 = go.Figure()

# Add traces for each year-month combination
for year in daily_sales['year'].unique():
    for month in daily_sales[daily_sales['year'] == year]['month'].unique():
        data = daily_sales[(daily_sales['year'] == year) & (daily_sales['month'] == month)]
        month_name = pd.to_datetime(f'{year}-{month}-01').strftime('%B %Y')
        
        fig2.add_trace(go.Scatter(
            x=data['day'],
            y=data['Id. Venta'],
            mode='lines+markers',
            name=month_name,
            hovertemplate='Day: %{x}<br>Sales: %{y}<br>Revenue: $%{customdata[0]:,.0f}<extra></extra>',
            customdata=data[['Precio']]
        ))

fig2.update_layout(
    title='Daily Sales Throughout the Month (Interactive)',
    xaxis_title='Day of Month',
    yaxis_title='Number of Sales',
    hovermode='x unified',
    legend_title='Month',
    template='plotly_white'
)

fig2.show()

# 3. Additional interactive chart: Hourly sales distribution
hourly_pattern = df.groupby(['year', 'month', 'hour']).agg({
    'Id. Venta': 'count',
    'Precio': 'sum'
}).reset_index()

fig3 = px.line(
    hourly_pattern,
    x='hour',
    y='Id. Venta',
    animation_frame='month',
    animation_group='hour',
    color='year',
    title='Hourly Sales Pattern by Month (Interactive)',
    labels={'hour': 'Hour of Day', 'Id. Venta': 'Number of Sales', 'year': 'Year'}
)

fig3.update_layout(
    xaxis=dict(tickmode='linear', tick0=0, dtick=1),
    hovermode='x unified'
)

fig3.show()

print("\n=== INTERACTIVE CHARTS CREATED ===")
print("1. Best Selling Times Heatmap: Shows revenue by hour and month")
print("2. Daily Sales Line Chart: Shows sales patterns throughout each month")
print("3. Hourly Sales Pattern: Shows how sales vary by hour across months")
print("\nHover over the charts to see detailed information!")

In [ ]:
# TOP 10 MOST SOLD PRODUCTS ANALYSIS

# Group by product to get total quantity sold and revenue
product_analysis = df.groupby('Producto').agg({
    'Cantidad': 'sum',  # Total quantity sold
    'Precio': 'sum'     # Total revenue
}).reset_index()

# Sort by quantity sold (descending) and get top 10
top_10_products = product_analysis.sort_values('Cantidad', ascending=False).head(10)

print("=== TOP 10 MOST SOLD PRODUCTS ===")
print(top_10_products[['Producto', 'Cantidad', 'Precio']].to_string(index=False))

# Create visualization for top 10 products
plt.figure(figsize=(16, 10))

# 1. Top 10 products by quantity sold
plt.subplot(2, 2, 1)
plt.bar(range(len(top_10_products)), top_10_products['Cantidad'], color='skyblue')
plt.title('Top 10 Products by Quantity Sold', fontsize=14, fontweight='bold')
plt.xlabel('Product')
plt.ylabel('Total Quantity Sold')
plt.xticks(range(len(top_10_products)), top_10_products['Producto'], rotation=45, ha='right')

# 2. Top 10 products by revenue
plt.subplot(2, 2, 2)
plt.bar(range(len(top_10_products)), top_10_products['Precio'], color='lightcoral')
plt.title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
plt.xlabel('Product')
plt.ylabel('Total Revenue ($)')
plt.xticks(range(len(top_10_products)), top_10_products['Producto'], rotation=45, ha='right')

# 3. Pie chart of product distribution by quantity
plt.subplot(2, 2, 3)
plt.pie(top_10_products['Cantidad'], labels=top_10_products['Producto'], autopct='%1.1f%%', startangle=90)
plt.title('Product Distribution by Quantity (Top 10)', fontsize=12, fontweight='bold')

# 4. Scatter plot: Quantity vs Revenue
plt.subplot(2, 2, 4)
plt.scatter(top_10_products['Cantidad'], top_10_products['Precio'], s=100, alpha=0.7, color='green')
plt.title('Quantity vs Revenue Correlation', fontsize=12, fontweight='bold')
plt.xlabel('Total Quantity Sold')
plt.ylabel('Total Revenue ($)')

# Add product labels to scatter plot
for i, row in top_10_products.iterrows():
    plt.annotate(row['Producto'], (row['Cantidad'], row['Precio']), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\n=== SUMMARY ===")
print(f"Total unique products: {df['Producto'].nunique()}")
print(f"Total sales records: {len(df)}")
print(f"\nMost popular product: {top_10_products.iloc[0]['Producto']}")
print(f"Quantity sold: {top_10_products.iloc[0]['Cantidad']}")
print(f"Revenue generated: ${top_10_products.iloc[0]['Precio']:,}")

# Interactive chart using Plotly
fig = go.Figure()

fig.add_trace(go.Bar(
    x=top_10_products['Producto'],
    y=top_10_products['Cantidad'],
    text=top_10_products['Cantidad'],
    textposition='outside',
    marker_color='skyblue',
    name='Quantity Sold'
))

fig.update_layout(
    title='Top 10 Most Sold Products',
    xaxis_title='Product',
    yaxis_title='Total Quantity Sold',
    xaxis_tickangle=-45,
    template='plotly_white'
)

fig.show()

print("\n=== RECOMMENDATIONS ===")
print("1. Focus marketing efforts on the top 3 most popular products")
print("2. Consider inventory optimization for high-volume products")
print("3. Analyze pricing strategy for products with high quantity but low revenue")
print("4. Investigate opportunities to cross-sell with top-performing products")